In [2]:
import polars as pl
from pathlib import Path
import numpy as np

In [7]:
x = np.linspace(-0.5, 0.5, 21)
xx, yy = np.meshgrid(x,x)
Z = xx + 1j*yy

In [14]:
Z[0]

array([-0.5 -0.5j, -0.45-0.5j, -0.4 -0.5j, -0.35-0.5j, -0.3 -0.5j,
       -0.25-0.5j, -0.2 -0.5j, -0.15-0.5j, -0.1 -0.5j, -0.05-0.5j,
        0.  -0.5j,  0.05-0.5j,  0.1 -0.5j,  0.15-0.5j,  0.2 -0.5j,
        0.25-0.5j,  0.3 -0.5j,  0.35-0.5j,  0.4 -0.5j,  0.45-0.5j,
        0.5 -0.5j])

In [2]:
cwd = Path.cwd().parent
df = pl.read_csv(cwd/'data/raw/CSI300.csv', schema_overrides={'':pl.Datetime})
df = df.rename({'':'timestamp'})

In [3]:
df.head()

timestamp,open,close,high,low,volume,money
datetime[μs],f64,f64,f64,f64,f64,f64
2010-01-04 09:31:00,3592.47,3596.3,3597.46,3575.68,1.160765e8,1.5548e9
2010-01-04 09:32:00,3595.09,3592.43,3595.81,3592.2,5.39317e7,7.65159e8
2010-01-04 09:33:00,3592.51,3588.8,3592.51,3588.79,4.3488e7,6.09674e8
2010-01-04 09:34:00,3588.37,3586.26,3588.37,3586.22,3.92227e7,5.72684e8
2010-01-04 09:35:00,3586.57,3586.12,3587.15,3585.9,3.7023e7,5.23588e8


In [4]:
df.describe()

statistic,timestamp,open,close,high,low,volume,money
str,str,f64,f64,f64,f64,f64,f64
"""count""","""858000""",858000.0,858000.0,858000.0,858000.0,858000.0,858000.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""","""2017-05-14 12:08:14.979021""",3490.108582,3490.116744,3490.920191,3489.297015,4.9451e7,7.1116e8
"""std""",null,804.792566,804.789877,804.999461,804.578656,5.0158e7,7.6483e8
"""min""","""2010-01-04 09:31:00""",2024.22,2024.25,2024.4,2023.33,0.0,0.0
"""25%""","""2013-09-10 14:01:00""",2910.99,2910.98,2911.67,2910.31,2.32991e7,2.78319104e8
"""50%""","""2017-05-16 13:01:00""",3464.86,3464.87,3465.54,3464.14,3.60328e7,4.96648192e8
"""75%""","""2021-01-13 10:30:00""",3966.09,3966.09,3966.94,3965.3,5.7395e7,8.70444381e8
"""max""","""2024-09-19 15:00:00""",5929.53,5929.01,5930.91,5922.07,3.7519e9,5.0001e10


In [5]:
df_5m = df.group_by_dynamic('timestamp', every='5m', closed='right', label='right').agg([
    pl.col('open').first(),
    pl.col('high').max(),
    pl.col('low').min(),
    pl.col('close').last(),
    pl.col('volume').sum()
])

In [6]:
df_5m.slice(40, 10)

timestamp,open,high,low,close,volume
datetime[μs],f64,f64,f64,f64,f64
2010-01-04 14:25:00,3560.32,3560.65,3555.59,3556.63,1.121188e8
2010-01-04 14:30:00,3556.81,3558.05,3555.72,3556.6,1.052544e8
2010-01-04 14:35:00,3556.11,3556.11,3551.5,3551.5,1.233648e8
2010-01-04 14:40:00,3551.43,3551.48,3547.29,3547.75,1.38822e8
2010-01-04 14:45:00,3547.46,3547.53,3545.76,3547.22,1.572068e8
2010-01-04 14:50:00,3547.37,3553.0,3547.37,3548.6,1.602184e8
2010-01-04 14:55:00,3548.91,3548.91,3540.96,3540.96,2.093308e8
2010-01-04 15:00:00,3541.04,3541.04,3535.23,3535.23,2.75202e8
2010-01-05 09:35:00,3545.79,3555.25,3545.18,3545.18,1.761138e8


$$ r_{i} = ln(S_{i}/S_{i-1})$$

$$ rv = \|{r_{i}}\| $$

In [7]:
df_5m_rv = df_5m.with_columns((pl.col('close')/pl.col('close').shift(1)).log().abs().alias('rv').fill_null(0))

In [8]:
df_5m_rv

timestamp,open,high,low,close,volume,rv
datetime[μs],f64,f64,f64,f64,f64,f64
2010-01-04 09:35:00,3592.47,3597.46,3575.68,3586.12,2.897419e8,0.0
2010-01-04 09:40:00,3586.54,3587.02,3577.52,3578.67,1.97582e8,0.00208
2010-01-04 09:45:00,3577.93,3583.88,3576.62,3583.45,1.668742e8,0.001335
2010-01-04 09:50:00,3583.56,3583.97,3574.92,3574.92,1.84303e8,0.002383
2010-01-04 09:55:00,3574.72,3574.72,3563.52,3568.06,2.059669e8,0.001921
…,…,…,…,…,…,…
2024-09-19 14:40:00,3199.15,3199.57,3195.35,3195.35,1.627192e8,0.001182
2024-09-19 14:45:00,3195.37,3195.37,3193.02,3193.66,1.738635e8,0.000529
2024-09-19 14:50:00,3193.68,3195.54,3193.44,3195.44,1.924572e8,0.000557


$$ RV = \sqrt {(\frac{\sum ((rv)_{i}^{2})}{N})} $$

In [23]:
def with_interval(df, interval):
    '''
    df:
    polars.DataFrame
    
    interval:
    [5m, 30m, 1d, 1w]
    '''
    if interval == '1w':
        start = 'monday'
    else:
        start = 'window'
    
    return df.group_by_dynamic('timestamp', every=interval, closed='right', label='right', start_by=start).agg([
        pl.col('open').first(),
        pl.col('high').max(),
        pl.col('low').min(),
        pl.col('close').last(),
        pl.col('volume').sum(),
        (pl.col('rv').pow(2).sum() / pl.len()).sqrt()
        ])

In [37]:
exp_df = with_interval(df_5m_rv, '1d')

In [38]:
exp_df.slice(25,10)

timestamp,open,high,low,close,volume,rv
datetime[μs],f64,f64,f64,f64,f64,f64
2010-02-09 00:00:00,3152.39,3172.15,3133.97,3150.99,3.7222e9,0.001692
2010-02-10 00:00:00,3146.83,3178.78,3144.48,3169.19,3.5854e9,0.001385
2010-02-11 00:00:00,3198.07,3214.46,3182.92,3214.13,3.4059e9,0.001478
2010-02-12 00:00:00,3216.39,3238.3,3207.27,3220.4,3.5670e9,0.001038
2010-02-13 00:00:00,3234.88,3253.08,3228.91,3251.28,3.3956e9,0.000795
2010-02-23 00:00:00,3248.68,3261.95,3232.78,3233.34,3.7927e9,0.001436
2010-02-24 00:00:00,3223.03,3223.07,3152.99,3198.63,4.2661e9,0.002105
2010-02-25 00:00:00,3175.94,3244.82,3165.38,3244.48,4.9810e9,0.001909
2010-02-26 00:00:00,3252.87,3293.95,3249.36,3292.13,6.7250e9,0.00137


In [26]:
exp_df.describe()

statistic,timestamp,open,high,low,close,volume,rv
str,str,f64,f64,f64,f64,f64,f64
"""count""","""3575""",3575.0,3575.0,3575.0,3575.0,3575.0,3575.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""","""2017-05-14 23:52:44.979021""",3488.65962,3516.685019,3461.094321,3490.938834,1.1868e10,0.001586
"""std""",null,805.24082,810.546078,797.621223,804.983336,7.9695e9,0.000962
"""min""","""2010-01-05 00:00:00""",2079.87,2118.76,2023.33,2086.97,2.1824e9,0.000371
"""25%""","""2013-09-12 00:00:00""",2914.01,2935.6,2886.02,2917.28,6.9453e9,0.001024
"""50%""","""2017-05-17 00:00:00""",3462.61,3488.13,3441.26,3466.35,1.0119e10,0.001354
"""75%""","""2021-01-14 00:00:00""",3962.13,3991.69,3936.48,3968.2,1.4293e10,0.001827
"""max""","""2024-09-20 00:00:00""",5922.07,5930.91,5747.66,5807.72,6.7939e10,0.012535
